
# Problem 5 - Value Iteration on the 5x5 gridworld (EE 5531 Assignment-2).

Same environment as Problem 4. The only change to the sweep is that the
average over actions becomes a max over actions:

    v_{k+1}(s) = max_a sum_s' p(s'|s,a) [ r + gamma v_k(s') ]

Transitions are deterministic here, so the inner sum collapses to one term.


In [1]:
import time
import numpy as np

In [2]:
import matplotlib
matplotlib.use("Agg")           # headless; drop this line when running locally
from Grid_show_actions import plot_actions

In [3]:
GAMMA = 0.95
REWARD = -1.0
TERMINALS = {(0, 0), (4, 4)}

In [4]:
# Order matters: the template's arrow dictionary uses keys like "RD", "RLU",
# always in the order R, L, U, D. Building strings in this order keeps them valid.
ACTIONS = {"R": (0, 1), "L": (0, -1), "U": (-1, 0), "D": (1, 0)}

In [5]:
def step(row, col, action):
    dr, dc = ACTIONS[action]
    next_row, next_col = row + dr, col + dc
    if next_row < 0 or next_row > 4 or next_col < 0 or next_col > 4:
        return row, col
    return next_row, next_col

In [6]:
def value_iteration(gamma=GAMMA, theta=1e-8, max_sweeps=10000):
    values = np.zeros((5, 5))
    history = [values.copy()]
    deltas = []

    for sweep in range(1, max_sweeps + 1):
        new_values = np.zeros((5, 5))

        for row in range(5):
            for col in range(5):
                if (row, col) in TERMINALS:
                    continue

                best = -np.inf
                for action in ACTIONS:
                    next_row, next_col = step(row, col, action)
                    q = REWARD + gamma * values[next_row, next_col]
                    best = max(best, q)
                new_values[row, col] = best

        delta = np.abs(new_values - values).max()
        values = new_values
        history.append(values.copy())
        deltas.append(delta)

        if delta < theta:
            break

    return values, history, deltas, sweep

In [7]:
def greedy_policy(values, gamma=GAMMA, tol=1e-9):
    """Action string per state, listing every action that attains the max.

    Ties are kept rather than broken. Comparing floats with a tolerance matters:
    exact == would miss ties that are equal only up to rounding.
    """
    policy = []
    for row in range(5):
        policy_row = []
        for col in range(5):
            if (row, col) in TERMINALS:
                policy_row.append("Nothing")   # template skips 7-char entries
                continue

            q = {}
            for action in ACTIONS:
                next_row, next_col = step(row, col, action)
                q[action] = REWARD + gamma * values[next_row, next_col]

            best = max(q.values())
            # ACTIONS is ordered R, L, U, D, so the string comes out valid
            label = "".join(a for a in ACTIONS if q[a] > best - tol)
            policy_row.append(label)
        policy.append(policy_row)
    return policy

In [8]:
def closed_form(gamma=GAMMA):
    
    """v*(s) = -(1 - gamma^d)/(1 - gamma), d = steps to the nearest terminal.

    Optimal play walks a shortest path, so the return is a geometric series of
    d discounted -1 rewards. Terminals are at (0,0) and (4,4), so with only
    4-directional moves d is the smaller of the two Manhattan distances.
    """
    
    v = np.zeros((5, 5))
    for row in range(5):
        for col in range(5):
            d = min(row + col, (4 - row) + (4 - col))
            v[row, col] = -(1 - gamma ** d) / (1 - gamma)
    return v

In [9]:
def show(values, title):
    print(title)
    for row in range(5):
        print("  " + "  ".join(f"{values[row, col]:8.4f}" for col in range(5)))
    print()

In [10]:
if __name__ == "__main__":
    start = time.perf_counter()
    values, history, deltas, sweeps = value_iteration()
    elapsed = time.perf_counter() - start

    print(f"gamma = {GAMMA}, converged after {sweeps} sweeps "
          f"(theta = 1e-8) in {elapsed * 1000:.2f} ms\n")

    for k in (1, 2, 3, 4, 5):
        if k < len(history):
            show(history[k], f"after sweep {k}:")

    show(values, "optimal values v*:")

    exact = closed_form()
    show(exact, "closed form -(1 - gamma^d)/(1 - gamma):")
    print(f"max difference: {np.abs(values - exact).max():.3e}\n")

    print("delta per sweep:")
    print("  " + ", ".join(f"{d:.4f}" for d in deltas) + "\n")

    policy = greedy_policy(values)
    print("optimal policy (all tied actions listed):")
    for row in policy:
        print("  " + "  ".join(f"{cell:>7}" for cell in row))
    print()

    # The template's arrow dictionary is missing one of the 15 combinations.
    arrow_keys = {"R", "L", "U", "D", "RD", "RU", "RL", "LD", "LU", "UD",
                  "RLU", "RLD", "LUD", "RLUD"}
    used = {cell for row in policy for cell in row if cell != "Nothing"}
    missing = used - arrow_keys
    print(f"action labels used: {sorted(used)}")
    print(f"labels not in the template's arrow dict: {sorted(missing) or 'none'}\n")

    assert values[0, 0] == 0.0 and values[4, 4] == 0.0
    assert np.allclose(values, exact, atol=1e-6)
    assert np.allclose(values, values[::-1, ::-1].T), "anti-diagonal symmetry"
    print("checks passed: terminals 0, matches closed form, symmetric\n")

    plot_actions(policy)
    print("policy plot written to plot.png")

gamma = 0.95, converged after 5 sweeps (theta = 1e-8) in 0.36 ms

after sweep 1:
    0.0000   -1.0000   -1.0000   -1.0000   -1.0000
   -1.0000   -1.0000   -1.0000   -1.0000   -1.0000
   -1.0000   -1.0000   -1.0000   -1.0000   -1.0000
   -1.0000   -1.0000   -1.0000   -1.0000   -1.0000
   -1.0000   -1.0000   -1.0000   -1.0000    0.0000

after sweep 2:
    0.0000   -1.0000   -1.9500   -1.9500   -1.9500
   -1.0000   -1.9500   -1.9500   -1.9500   -1.9500
   -1.9500   -1.9500   -1.9500   -1.9500   -1.9500
   -1.9500   -1.9500   -1.9500   -1.9500   -1.0000
   -1.9500   -1.9500   -1.9500   -1.0000    0.0000

after sweep 3:
    0.0000   -1.0000   -1.9500   -2.8525   -2.8525
   -1.0000   -1.9500   -2.8525   -2.8525   -2.8525
   -1.9500   -2.8525   -2.8525   -2.8525   -1.9500
   -2.8525   -2.8525   -2.8525   -1.9500   -1.0000
   -2.8525   -2.8525   -1.9500   -1.0000    0.0000

after sweep 4:
    0.0000   -1.0000   -1.9500   -2.8525   -3.7099
   -1.0000   -1.9500   -2.8525   -3.7099   -2.8525
   -

D:\SEM 7\Reinforcement_Learning\Assignment_2\Grid_show_actions.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
